In [ ]:
# Install required libraries
!pip -q install pandas numpy scikit-learn transformers datasets accelerate torch seaborn matplotlib tqdm

In [ ]:
# Imports and reproducibility
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import warnings
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [ ]:
# Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/multisocial_outputs'
TEST_PATH = os.path.join(BASE_DIR, 'multisocial_test.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'pretrained')
RESULTS_JSON = os.path.join(OUTPUT_DIR, 'results_pretrained_multilingual.json')

# Using a single model for all languages as requested
MODEL_NAME = 'roberta-base-openai-detector'

# CSV output directories
RESULTS_DIR = '/content/results'
DRIVE_RESULTS_DIR = os.path.join(BASE_DIR, 'pretrained')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

assert os.path.exists(TEST_PATH), f'Test file not found: {TEST_PATH}'
print(f'Test path: {TEST_PATH}')

Mounted at /content/drive
Test path: /content/drive/MyDrive/multisocial_outputs/multisocial_test.csv


In [ ]:
# Load and validate test data
test_df = pd.read_csv(TEST_PATH)
required_cols = {'text', 'label', 'language'}
missing = required_cols - set(test_df.columns)
if missing:
    raise ValueError(f'Missing required columns in test CSV: {sorted(missing)}')

test_df = test_df.dropna(subset=['text', 'label', 'language']).copy()
test_df['text'] = test_df['text'].astype(str)
test_df['label'] = test_df['label'].astype(int)
test_df['language'] = test_df['language'].astype(str)

assert not test_df.empty, 'Test dataframe is empty after cleanup.'

print('Samples per language:')
print(test_df['language'].value_counts())

for lang in ['en', 'vi', 'zh', 'ar']:
    lang_count = int((test_df['language'] == lang).sum())
    print(f'Language {lang}: {lang_count} rows')
    assert lang_count > 0, f'No test samples found for language: {lang}'

Samples per language:
language
en    800
zh    800
ar    800
vi    797
Name: count, dtype: int64
Language en: 800 rows
Language vi: 797 rows
Language zh: 800 rows
Language ar: 800 rows


In [ ]:
# Helper functions
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()
    return tokenizer, model


def logits_to_prob_machine(logits: torch.Tensor) -> torch.Tensor:
    # Supports either 1-logit or 2-logit classifier heads.
    if logits.ndim == 2 and logits.shape[-1] == 1:
        probs = torch.sigmoid(logits.squeeze(-1))
    elif logits.ndim == 2 and logits.shape[-1] == 2:
        machine_logit = logits[:, 1] - logits[:, 0]
        probs = torch.sigmoid(machine_logit)
    else:
        raise ValueError(f'Unexpected logits shape: {tuple(logits.shape)}')
    return probs


def predict_probs(texts, tokenizer, model, batch_size=32, max_length=256):
    probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Inference'):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            out = model(**enc, return_dict=True)
            batch_probs = logits_to_prob_machine(out.logits)

        probs.extend(batch_probs.detach().cpu().numpy().tolist())

    probs = np.array(probs, dtype=np.float32)
    assert len(probs) == len(texts), 'Probability output length mismatch.'
    return probs


def evaluate_language(df_lang: pd.DataFrame):
    y_true = df_lang['label'].values.astype(int)
    y_prob = df_lang['prob_machine'].values.astype(np.float32)
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro')
    return float(acc), float(f1m), int(len(df_lang))


def safe_roc_auc(y_true, y_score):
    """Compute ROC-AUC, return NaN with warning if single class."""
    if len(np.unique(y_true)) < 2:
        warnings.warn(f'Only one class in y_true, ROC-AUC undefined. Returning NaN.')
        return float('nan')
    return float(roc_auc_score(y_true, y_score))


def compute_classification_metrics(y_true, y_prob, threshold=0.5):
    """Compute accuracy, precision_macro, recall_macro, f1_macro, roc_auc."""
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'roc_auc': safe_roc_auc(y_true, y_prob),
    }


def save_df(df, path, index=False):
    """Save DataFrame to CSV and print confirmation."""
    df.to_csv(path, index=index)
    print(f'Saved: {path} ({len(df)} rows)')

In [ ]:
# Evaluate single model for all languages
print(f'Evaluating all languages using: {MODEL_NAME}')

tokenizer, model = load_model_and_tokenizer(MODEL_NAME)
all_probs = predict_probs(test_df['text'].tolist(), tokenizer, model, batch_size=32, max_length=256)
test_df['prob_machine'] = all_probs

# Cleanup
del model
del tokenizer
clear_gpu_memory()
print('Inference complete and model freed from memory.')

assert test_df['prob_machine'].notna().all(), 'NaN probabilities detected in output.'

Evaluating all languages using: roberta-base-openai-detector


config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base-openai-detector
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Inference:   0%|          | 0/100 [00:00<?, ?it/s]

Inference complete and model freed from memory.


In [ ]:
# Compute per-language and overall metrics
y_true_all = test_df['label'].values.astype(int)
y_prob_all = test_df['prob_machine'].values.astype(np.float32)

# Per-language
results = []
for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = test_df[test_df['language'] == lang].copy()
    assert not lang_df.empty, f'No prediction rows for language: {lang}'

    y_true = lang_df['label'].values.astype(int)
    y_prob = lang_df['prob_machine'].values.astype(np.float32)
    metrics = compute_classification_metrics(y_true, y_prob)
    metrics['language'] = lang
    metrics['n_samples'] = int(len(lang_df))
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df[['language', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc', 'n_samples']]

# Overall (all languages combined)
overall_metrics = compute_classification_metrics(y_true_all, y_prob_all)
overall_metrics['language'] = 'overall'
overall_metrics['n_samples'] = int(len(test_df))
overall_df = pd.DataFrame([overall_metrics])
overall_df = overall_df[['language', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc', 'n_samples']]

# Display combined summary
full_results_df = pd.concat([overall_df, results_df], ignore_index=True)
print('Pretrained baseline metrics (roberta-base-openai-detector):')
display(full_results_df)

assert len(results_df) == 4, 'Expected metrics for 4 languages.'
assert (results_df['n_samples'] > 0).all(), 'One or more languages have zero samples.'

Pretrained baseline metrics (roberta-base-openai-detector):


,language,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,n_samples
0,overall,0.526744,0.547810,0.530081,0.480270,0.589987,3197
1,en,0.641250,0.674686,0.641250,0.623221,0.737525,800
2,vi,0.506901,0.517618,0.515168,0.492994,0.516240,797
3,zh,0.476250,0.455540,0.476250,0.407220,0.481178,800
4,ar,0.482500,0.349656,0.482500,0.335772,0.616412,800


In [ ]:
# Save CSV exports to local and Drive
# 1. Overall metrics
save_df(overall_df, os.path.join(RESULTS_DIR, 'pretrained_overall_metrics.csv'))
save_df(overall_df, os.path.join(DRIVE_RESULTS_DIR, 'pretrained_overall_metrics.csv'))

# 2. Per-language metrics
save_df(results_df, os.path.join(RESULTS_DIR, 'pretrained_per_language_metrics.csv'))
save_df(results_df, os.path.join(DRIVE_RESULTS_DIR, 'pretrained_per_language_metrics.csv'))

# 3. Full predictions with labels
pred_df = test_df[['text', 'label', 'language', 'prob_machine']].copy()
pred_df['pred_label'] = (pred_df['prob_machine'] >= 0.5).astype(int)
save_df(pred_df, os.path.join(RESULTS_DIR, 'pretrained_predictions.csv'))
save_df(pred_df, os.path.join(DRIVE_RESULTS_DIR, 'pretrained_predictions.csv'))

print('All CSV exports complete.')

Saved: /content/results/pretrained_overall_metrics.csv (1 rows)
Saved: /content/drive/MyDrive/multisocial_outputs/pretrained/pretrained_overall_metrics.csv (1 rows)
Saved: /content/results/pretrained_per_language_metrics.csv (4 rows)
Saved: /content/drive/MyDrive/multisocial_outputs/pretrained/pretrained_per_language_metrics.csv (4 rows)
Saved: /content/results/pretrained_predictions.csv (3197 rows)
Saved: /content/drive/MyDrive/multisocial_outputs/pretrained/pretrained_predictions.csv (3197 rows)
All CSV exports complete.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm_paths = {}
for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = test_df[test_df['language'] == lang].copy()
    # Convert probabilities to binary predictions
    y_true = lang_df['label'].values
    y_pred = (lang_df['prob_machine'].values >= 0.5).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['human (0)', 'machine (1)'],
        yticklabels=['human (0)', 'machine (1)'],
    )
    plt.title(f'Confusion Matrix - {lang}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()

    cm_path = os.path.join(OUTPUT_DIR, f'cm_{lang}.png')
    plt.savefig(cm_path, dpi=200)
    plt.close()

    cm_paths[lang] = cm_path
    assert os.path.exists(cm_path), f'Confusion matrix not saved: {cm_path}'

print('Saved confusion matrices:')
for lang, path in cm_paths.items():
    print(f'  {lang}: {path}')

Saved confusion matrices:
  en: /content/drive/MyDrive/multisocial_outputs/pretrained/cm_en.png
  vi: /content/drive/MyDrive/multisocial_outputs/pretrained/cm_vi.png
  zh: /content/drive/MyDrive/multisocial_outputs/pretrained/cm_zh.png
  ar: /content/drive/MyDrive/multisocial_outputs/pretrained/cm_ar.png


In [ ]:
# Plot combined ROC curve per language
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

roc_plot_path = os.path.join(OUTPUT_DIR, 'roc_combined.png')

plt.figure(figsize=(7, 6))
for lang in ['en', 'vi', 'zh', 'ar']:
    lang_df = test_df[test_df['language'] == lang].copy()
    y_true = lang_df['label'].values.astype(int)
    y_prob = lang_df['prob_machine'].values.astype(np.float32)
    auc_val = safe_roc_auc(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'{lang} (AUC={auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.7)
plt.title(f'Combined ROC Curves — {MODEL_NAME}')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(roc_plot_path, dpi=220)
plt.close()

assert os.path.exists(roc_plot_path), f'ROC plot not saved: {roc_plot_path}'
print(f'Saved ROC plot: {roc_plot_path}')

Saved ROC plot: /content/drive/MyDrive/multisocial_outputs/pretrained/roc_combined.png


In [ ]:
# Save results
payload = {
    'seed': SEED,
    'model_used': MODEL_NAME,
    'overall': overall_df.to_dict(orient='records')[0],
    'per_language': results_df.to_dict(orient='records'),
    'roc_plot_path': roc_plot_path,
    'cm_paths': cm_paths,
}

with open(RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)

assert os.path.exists(RESULTS_JSON), f'Results file was not created: {RESULTS_JSON}'
print(f'Saved pretrained results to: {RESULTS_JSON}')
print('Done.')

Saved pretrained results to: /content/drive/MyDrive/multisocial_outputs/pretrained/results_pretrained_multilingual.json
Done.
